# 01 · Exploración de mis entrenamientos de Hevy

**Objetivo:** convertir los datos en bruto de Hevy en una tabla limpia y empezar a hacerle preguntas.

En ingeniería de IA, esta parte (entender y limpiar los datos) suele ser el 70-80 % del trabajo.
Un LLM solo puede darte buenas recomendaciones si le pasas buenos datos: *garbage in, garbage out*.

> Antes de ejecutar: `python -m src.download_workouts` para tener los datos en `data/`.

In [ ]:
import sys
from pathlib import Path

# Permite importar el paquete `src` desde la carpeta notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.dataset import load_sets_df

# Estilo sobrio para las gráficas
COLOR = "#2a78d6"
plt.rcParams.update({
    "figure.figsize": (10, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7",
    "axes.grid": True,
    "grid.color": "#ecebe7",
    "axes.axisbelow": True,
})

## 1. Cargar los datos: una fila por serie

Hevy devuelve una estructura anidada: **entrenamiento → ejercicios → series**.
`load_sets_df()` la *aplana* en una tabla donde cada fila es una serie. Es la granularidad más fina,
y desde ella podemos agregar como queramos (por sesión, semana, ejercicio, músculo...).

Columnas calculadas que añadimos (*feature engineering*):
- `volume_kg` = peso × repeticiones
- `e1rm_kg` = 1RM estimado (fórmula de Epley), para comparar series con distinto peso y reps
- `week` = lunes de la semana, para agrupar por semanas

In [ ]:
df = load_sets_df(ROOT / "data" / "hevy_workouts.json", ROOT / "data" / "hevy_exercise_templates.json")
print(df.shape)
df.head()

## 2. Calidad de los datos

Antes de analizar nada, siempre comprobamos: ¿qué tipos hay? ¿qué falta? ¿hay valores raros?

In [ ]:
df[["weight_kg", "reps", "rpe", "duration_seconds", "distance_meters"]].isna().mean().round(3).rename("% vacíos")

In [ ]:
df["set_type"].value_counts()

In [ ]:
# Ejercicios sin peso: peso corporal (dominadas, fondos...) o por tiempo (plancha...)
df[df["weight_kg"].isna()].groupby("exercise").size().sort_values(ascending=False).head(10)

**Fíjate:** las series sin peso tienen `volume_kg = 0`. Eso hace que el volumen en kg
*infravalore* ejercicios como dominadas o fondos. Por eso, para medir el trabajo por músculo,
contaremos **series** en vez de kg (es además lo que usa la ciencia del entrenamiento para hipertrofia).

## 3. Visión general

In [ ]:
sessions = df.groupby("workout_id").agg(
    date=("date", "first"),
    title=("workout_title", "first"),
    start=("start_time", "first"),
    end=("end_time", "first"),
    sets=("set_index", "size"),
    volume_kg=("volume_kg", "sum"),
)
sessions["duration_min"] = (sessions["end"] - sessions["start"]).dt.total_seconds() / 60

print(f"Entrenamientos: {len(sessions)}")
print(f"Desde {sessions['date'].min()} hasta {sessions['date'].max()}")
print(f"Duración media: {sessions['duration_min'].mean():.0f} min")
print(f"Series por sesión (media): {sessions['sets'].mean():.1f}")
sessions["title"].value_counts().head(10)

### ⚠️ Valores atípicos (*outliers*)

La duración media sale sospechosamente alta. Miremos la distribución:

In [ ]:
sessions["duration_min"].describe().round(0)

Hay sesiones de más de 20 horas: son entrenamientos que **se quedaron abiertos en Hevy**.
La media es muy sensible a estos valores; la **mediana** no. Dos lecciones:

1. Mira siempre la distribución (`describe()`), no solo la media.
2. Decide una regla de limpieza explícita y documentada. Aquí: descartar duraciones > 4 h.

In [ ]:
MAX_DURATION_MIN = 240
valid = sessions["duration_min"] <= MAX_DURATION_MIN
print(f"Sesiones descartadas para duración: {(~valid).sum()}")
print(f"Duración media limpia: {sessions.loc[valid, 'duration_min'].mean():.0f} min | mediana: {sessions['duration_min'].median():.0f} min")

## 4. ¿Cuántas veces entreno por semana?

In [ ]:
per_week = df.groupby("week")["workout_id"].nunique()
# Rellenamos las semanas sin entrenar con 0 para que se vean los huecos
all_weeks = pd.date_range(per_week.index.min(), per_week.index.max(), freq="W-MON").date
per_week = per_week.reindex(all_weeks, fill_value=0)

fig, ax = plt.subplots()
ax.bar(per_week.index, per_week.values, width=5, color=COLOR)
ax.axhline(per_week.mean(), color="#52514e", linewidth=1, linestyle="--")
ax.annotate(f"media {per_week.mean():.1f}", (per_week.index[-1], per_week.mean()),
            xytext=(4, 4), textcoords="offset points", color="#52514e", fontsize=9)
ax.set_title("Entrenamientos por semana", loc="left")
plt.show()

## 5. Volumen semanal (kg movidos)

In [ ]:
weekly_volume = df[~df["is_warmup"]].groupby("week")["volume_kg"].sum().reindex(all_weeks, fill_value=0)

fig, ax = plt.subplots()
ax.plot(weekly_volume.index, weekly_volume.values, color=COLOR, linewidth=2)
ax.set_title("Volumen semanal (kg)", loc="left")
plt.show()

## 6. Series por grupo muscular (últimas 8 semanas)

Una referencia habitual para hipertrofia es **~10-20 series efectivas por músculo y semana**.
Esto ya es una *métrica accionable*: el tipo de dato que le pasaremos al coach de IA.

In [ ]:
last_8 = df[(df["start_time"] >= df["start_time"].max() - pd.Timedelta(weeks=8)) & ~df["is_warmup"]]
sets_per_week = (last_8.groupby("muscle_group").size() / 8).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(sets_per_week.index, sets_per_week.values, color=COLOR, height=0.7)
ax.axvspan(10, 20, color="#ecebe7", zorder=0)
ax.set_title("Series por semana y grupo muscular (zona gris: 10-20)", loc="left")
ax.grid(axis="y", visible=False)
plt.show()
sets_per_week.sort_values(ascending=False).round(1)

## 7. ¿Estoy progresando? 1RM estimado por ejercicio

Para cada sesión nos quedamos con la **mejor serie** (máximo e1RM) de un ejercicio.
Cambia `EXERCISE` para ver otros.

In [ ]:
top_exercises = df[df["weight_kg"].notna()].groupby("exercise")["workout_id"].nunique().sort_values(ascending=False)
top_exercises.head(10)

In [ ]:
EXERCISE = top_exercises.index[0]

progress = df[(df["exercise"] == EXERCISE) & ~df["is_warmup"]].groupby("date")["e1rm_kg"].max()
trend = progress.rolling(5, min_periods=1).mean()

fig, ax = plt.subplots()
ax.scatter(progress.index, progress.values, color=COLOR, s=16, alpha=0.4, label="mejor serie del día")
ax.plot(trend.index, trend.values, color=COLOR, linewidth=2, label="media móvil (5 sesiones)")
ax.set_title(f"{EXERCISE}: 1RM estimado (kg)", loc="left")
ax.legend(frameon=False)
plt.show()

change = trend.iloc[-1] - trend.iloc[0]
print(f"Primera sesión: {progress.index[0]} | Última: {progress.index[-1]}")
print(f"Tendencia: {trend.iloc[0]:.1f} kg -> {trend.iloc[-1]:.1f} kg ({change:+.1f} kg)")

## 8. Ejercicios para ti

Para aprender de verdad, intenta resolver tú estas preguntas (pídeme pistas si te atascas):

1. ¿Cuál es tu ejercicio con **mayor mejora en %** de e1RM entre el primer y el último mes?
2. ¿Qué día de la semana entrenas más? (pista: `df["start_time"].dt.day_name()`)
3. ¿Hay algún ejercicio **estancado**, sin mejora en las últimas 8 semanas?
4. ¿Cuánto duran de media tus sesiones de cada tipo (`workout_title`)?

**Siguiente paso:** convertir las mejores métricas de aquí en funciones de `src/` y
pasárselas a un LLM para que actúe como coach.